# 02 — KnowYourRights · Indian Legal Rights Agent

A multi-agent **plan → retrieve → grade → write** pipeline over a database of central Indian law, with Wikipedia + web for background and current procedure. Built on the OpenAI Agents SDK + NVIDIA NIM.

*General legal information, not legal advice.*

## §0 · Install

In [ ]:
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=False)

# openai-agents pulls a recent openai client; the rest power retrieval, web + Wikipedia search, and the UI.
pip("openai-agents>=0.17", "openai>=1.40",
    "ddgs", "lancedb", "sentence-transformers", "transformers", "torch",
    "gradio", "pydantic>=2", "pandas", "pyarrow", "numpy",
    "requests", "python-dotenv")
print("installs done")

## §1 · Config — every knob lives here

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

# ── NVIDIA NIM ────────────────────────────────────────────────────────────────
NIM_BASE_URL   = "https://integrate.api.nvidia.com/v1"
# Tool-calling-capable on NIM (Llama 3.1/3.2/3.3 + Mistral support OpenAI-style tools natively).
NIM_MODEL      = "nvidia/nemotron-3-super-120b-a12b"
NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY", "")

# ── Data (your unzipped bundle lives under ./data/) ─────────────────────────────
DB_PATH    = os.environ.get("LEGAL_DB_PATH",  "./data/legal_db")
TABLE      = "laws"
PARQUET    = os.environ.get("LEGAL_PARQUET",  "./data/chunks_metadata.parquet")
CACHE_FILE = os.environ.get("LEGAL_CACHE",    "./data/enrichment_cache.json")

# ── Embedder (LOCKED to the DB — do not change) ────────────────────────────────
EMBED_MODEL        = "BAAI/bge-m3"
MAX_SEQ_LEN        = 1024
EMBED_QUERY_PREFIX = ""
EMBED_DEVICE       = "auto"

# ── Reranker (swappable; independent of the embedder) ──────────────────────────
USE_RERANKER    = True
RERANK_PRIMARY  = "Alibaba-NLP/gte-reranker-modernbert-base"
RERANK_FALLBACK = "BAAI/bge-reranker-base"

# ── Retrieval (notebook-01 contract) ───────────────────────────────────────────
FETCH_K, TOP_K, RRF_K = 25, 5, 60
LOW_SCORE      = 0.05
CITE_MIN_SCORE = 0.20             # cheap pre-filter on DB sections before the LLM grader sees them
MMR_LAMBDA     = 0.6
TOOL_TEXT_CAP  = 2500
TOPK_MIN, TOPK_MAX = 2, 10

# ── NIM rate-limit handling (free tier ≈ 40 requests/min) ──────────────────────
# A legal question costs ~3 LLM calls (plan + grade + write); chit-chat ~2 (plan + reply).
NIM_RPM             = 30
RETRY_MAX           = 6
RETRY_INITIAL_DELAY = 2.0
RETRY_MAX_DELAY     = 30.0
RETRY_MULTIPLIER    = 2.0
RETRY_FINAL_SLEEP   = 20

# ── Web search (DuckDuckGo via ddgs) ────────────────────────────────────────────
WEB_MAX_RESULTS = 3
WEB_TIMEOUT     = 12
WEB_CACHE_TTL   = 1800
WEB_MAX_PER_MIN = 8

# ── Wikipedia (MediaWiki API, for plain-language background) ────────────────────
WIKI_MAX_RESULTS = 2
WIKI_TIMEOUT     = 10

# ── Agents (each small + single-purpose; short prompts beat one giant prompt) ───
PLAN_TEMP        = 0.1            # planner: classify intent + plan research (structured JSON)
GRADE_TEMP       = 0.0            # grader: keep only genuinely relevant sources (structured JSON)
WRITER_TEMP      = 0.3           # writer: compose the user-facing answer (free prose, streamed)
WRITER_MAX_TOKENS = 1100
CONCIERGE_TEMP   = 0.5           # concierge: greetings / capability / out-of-scope

CATEGORIES = [
    "Fundamental Rights", "Criminal & Police", "Consumer & Services",
    "Employment & Labour", "Family & Marriage", "Property & Housing",
    "Women & Children", "Privacy & Data", "Health & Medicine", "Education",
    "Environment", "Taxation & Finance", "Business & Companies", "Information & RTI",
    "Civil Procedure & Courts", "Transport & Motor", "Government & Administration", "Other",
]

print("config loaded | model:", NIM_MODEL, "| db:", DB_PATH)


config loaded | model: nvidia/nemotron-3-super-120b-a12b | db: ./data/legal_db


## §2 · Observability — tool/LLM-call tracking + per-turn context

In [2]:
import json, time, hashlib, asyncio, warnings, gc, uuid, re as _re
from collections import Counter
from dataclasses import dataclass, field
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

class ToolTracker:
    """Cumulative record of pipeline activity: which sources were queried, and LLM-call count."""
    def __init__(self):
        self.calls = Counter(); self.errors = Counter(); self.llm_calls = 0
    def record_call(self, name):  self.calls[name] += 1
    def record_error(self, name): self.errors[name] += 1
    def record_llm(self):         self.llm_calls += 1
    def snapshot(self):
        return {"calls": dict(self.calls), "errors": dict(self.errors), "llm_calls": self.llm_calls}
    def reset(self):
        self.__init__(); print("tracker reset")
    def report(self):
        names = sorted(set(self.calls) | set(self.errors))
        print("\n┌── KnowYourRights · cumulative source usage ───────────────────")
        if not names:
            print("│  (no retrieval yet)")
        for n in names:
            print(f"│  {n:<12} queries={self.calls.get(n,0):<3} errors={self.errors.get(n,0)}")
        print(f"│  LLM (NIM) calls total: {self.llm_calls}")
        print("└───────────────────────────────────────────────────────────────")

TRACKER = ToolTracker()

@dataclass
class AppContext:
    """Per-turn state. `sources` holds the vetted (graded-relevant) Candidates -> the UI panel."""
    tracker: ToolTracker
    sources: list = field(default_factory=list)

class RateLimiter:
    """Async min-interval gate. Spaces LLM calls under NIM's RPM ceiling -> prevents most 429s."""
    def __init__(self, rpm):
        self.min_interval = 60.0 / max(1, rpm)
        self._lock = asyncio.Lock(); self._last = 0.0
    async def wait(self):
        async with self._lock:
            dt = time.monotonic() - self._last
            if dt < self.min_interval:
                await asyncio.sleep(self.min_interval - dt)
            self._last = time.monotonic()

NIM_LIMITER = RateLimiter(NIM_RPM)

def _diff(before, after):
    return {k: after[k] - before.get(k, 0) for k in after if after[k] - before.get(k, 0) > 0}

print("observability ready · ToolTracker + AppContext(sources) + RateLimiter(", NIM_RPM, "rpm )")


observability ready · ToolTracker + AppContext(sources) + RateLimiter( 30 rpm )


## §3 · Connect to NVIDIA NIM (throttled model + header-aware retry)

In [3]:
from openai import AsyncOpenAI, RateLimitError
from agents import (
    Agent, Runner, RunHooks, RunContextWrapper, function_tool,
    OpenAIChatCompletionsModel, ModelSettings,
    set_default_openai_client, set_default_openai_api, set_tracing_disabled,
)
from agents.model_settings import ModelRetrySettings, ModelRetryBackoffSettings
from agents.exceptions import MaxTurnsExceeded

assert NVIDIA_API_KEY, "Set NVIDIA_API_KEY (export NVIDIA_API_KEY=nvapi-... or put it in a .env file)."

# NIM is OpenAI-compatible but not OpenAI -> don't ship traces to OpenAI's backend.
set_tracing_disabled(True)
nim_client = AsyncOpenAI(base_url=NIM_BASE_URL, api_key=NVIDIA_API_KEY, max_retries=0, timeout=60.0)
set_default_openai_client(nim_client, use_for_tracing=False)
set_default_openai_api("chat_completions")   # NIM speaks chat-completions, not the Responses API

class ThrottledNIMModel(OpenAIChatCompletionsModel):
    """Wraps the chat-completions model to (1) space calls under NIM's RPM ceiling and
    (2) count every LLM call into the tracker. Both get_response and the streaming path are covered."""
    async def get_response(self, *args, **kwargs):
        await NIM_LIMITER.wait(); TRACKER.record_llm()
        return await super().get_response(*args, **kwargs)
    async def stream_response(self, *args, **kwargs):
        await NIM_LIMITER.wait(); TRACKER.record_llm()
        async for event in super().stream_response(*args, **kwargs):
            yield event

llm_model = ThrottledNIMModel(model=NIM_MODEL, openai_client=nim_client)

# Model-level 429 policy: header-aware, exponential backoff + jitter.
RETRY_SETTINGS = ModelRetrySettings(
    max_retries=RETRY_MAX,
    backoff=ModelRetryBackoffSettings(
        initial_delay=RETRY_INITIAL_DELAY, max_delay=RETRY_MAX_DELAY,
        multiplier=RETRY_MULTIPLIER, jitter=True,
    ),
)
print("NIM model ready:", NIM_MODEL, "| throttle:", NIM_RPM, "rpm | retries:", RETRY_MAX)

NIM model ready: nvidia/nemotron-3-super-120b-a12b | throttle: 30 rpm | retries: 6


In [4]:
# ── Smoke test (one LLM call) ──────────────────────────────────────────────────
_smoke = Agent(
    name="smoke-test",
    instructions="Reply with exactly: CONNECTION OK",
    model=llm_model,
    model_settings=ModelSettings(temperature=0, retry=RETRY_SETTINGS),
)
_r = await Runner.run(_smoke, "ping", max_turns=2)
print("smoke test ->", _r.final_output)
del _smoke

smoke test -> CONNECTION OK


## §4 · Load the legal database

In [5]:
import lancedb
from sentence_transformers import SentenceTransformer
import torch

# clear any prior objects so the cell is safe to re-run
for _v in ["emb", "db", "tbl", "chunks_df", "enrich_cache"]:
    if _v in globals(): del globals()[_v]
gc.collect()

_edev = ("cuda" if torch.cuda.is_available() else "cpu") if EMBED_DEVICE == "auto" else EMBED_DEVICE
emb = SentenceTransformer(EMBED_MODEL, trust_remote_code=True, device=_edev)
emb.max_seq_length = MAX_SEQ_LEN

# Resolve paths: honour the configured/env path if it exists, else look in ./ and ./data/,
# else search the working tree. Lets the notebook run from anywhere in the repo.
def _resolve(path, want_dir):
    check = os.path.isdir if want_dir else os.path.isfile
    if check(path):
        return path
    name = os.path.basename(path.rstrip("/"))
    for cand in (f"./{name}", f"./data/{name}"):
        if check(cand):
            return cand
    import glob
    for cand in glob.glob(f"./**/{name}", recursive=True):
        if check(cand):
            return cand
    return path  # unchanged -> the assert below gives a clear error

DB_PATH    = _resolve(DB_PATH,    want_dir=True)
PARQUET    = _resolve(PARQUET,    want_dir=False)
CACHE_FILE = _resolve(CACHE_FILE, want_dir=False)
assert os.path.isdir(DB_PATH),  f"LanceDB dir not found: {DB_PATH}  (set LEGAL_DB_PATH or fix §1)."
assert os.path.isfile(PARQUET), f"parquet not found: {PARQUET}  (set LEGAL_PARQUET or fix §1)."
print("resolved | DB_PATH =", DB_PATH, "| PARQUET =", PARQUET)

db  = lancedb.connect(DB_PATH)
tbl = db.open_table(TABLE)
chunks_df = pd.read_parquet(PARQUET)
enrich_cache = json.load(open(CACHE_FILE, encoding="utf-8")) if os.path.exists(CACHE_FILE) else {}

print(f"DB: {tbl.count_rows():,} rows | parquet: {len(chunks_df):,} chunks | embedder on {_edev}")
print("columns:", list(chunks_df.columns))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

resolved | DB_PATH = ./data/legal_db | PARQUET = ./data/chunks_metadata.parquet
DB: 38,890 rows | parquet: 38,890 chunks | embedder on cuda
columns: ['source_type', 'act_title', 'act_short_name', 'act_year', 'act_number', 'section_label', 'section_num', 'section_name', 'chapter', 'jurisdiction', 'status', 'effective_date', 'full_text', 'source_snapshot', 'unit_id', 'chunk_id', 'chunk_text', 'category', 'citation', 'embed_text']


## §5 · Reranker (GPU → CPU, with fallback)

In [7]:
RERANK_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
_rt = _rm = None
if USE_RERANKER:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

def get_reranker():
    """Lazy-load the reranker once, with automatic fallback. Raises if none can load."""
    global _rt, _rm
    if _rm is None:
        for name in [RERANK_PRIMARY, RERANK_FALLBACK]:
            try:
                _rt = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
                _rm = (AutoModelForSequenceClassification
                       .from_pretrained(name, trust_remote_code=True).to(RERANK_DEVICE).eval())
                print("reranker loaded:", name, "on", RERANK_DEVICE); break
            except Exception as ex:
                print("reranker load failed:", name, "->", str(ex)[:120]); _rt = _rm = None
        if _rm is None:
            raise RuntimeError("no reranker could be loaded")
    return _rt, _rm

def _rr_maxlen(m):
    cfg = m.config
    mp = int(getattr(cfg, "max_position_embeddings", 512) or 512)
    if getattr(cfg, "model_type", "") in ("xlm-roberta", "roberta", "camembert"):
        mp -= 2                          # RoBERTa-family position offset
    return max(8, min(mp, MAX_SEQ_LEN))

def rr_scores(query, docs):
    """Cross-encoder relevance scores in [0,1] for (query, doc) pairs."""
    t, m = get_reranker()
    ml = _rr_maxlen(m)
    with torch.no_grad():
        enc = t([[query, d] for d in docs], padding=True, truncation=True,
                max_length=ml, return_tensors="pt")
        enc.pop("token_type_ids", None)  # XLM-R / RoBERTa rerankers don't use segment ids
        enc = {k: v.to(RERANK_DEVICE) for k, v in enc.items()}
        return torch.sigmoid(m(**enc).logits.view(-1)).float().cpu().numpy()

if USE_RERANKER:
    try:
        _ = rr_scores("test query", ["a short document about testing"])
        print("reranker OK | score scale ~[0,1]")
    except Exception as e:
        print("reranker unavailable -> retrieval will fall back to RRF scores:", str(e)[:100])

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

reranker loaded: Alibaba-NLP/gte-reranker-modernbert-base on cuda:0
reranker OK | score scale ~[0,1]


## §6 · Retrieval engine — hybrid + rerank + MMR (the notebook-01 contract)

In [8]:
def rrf(lists, k=RRF_K):
    """Reciprocal Rank Fusion -> (ordered chunk_ids, score map)."""
    s = {}
    for ranks in lists:
        for p, cid in enumerate(ranks):
            s[cid] = s.get(cid, 0.0) + 1.0 / (k + p + 1)
    return sorted(s, key=s.get, reverse=True), s

def mmr(cand_vecs, base, lam=MMR_LAMBDA, k=TOP_K):
    """Maximal Marginal Relevance ordering for diversity vs relevance."""
    chosen, rest = [], list(range(len(base)))
    while rest and len(chosen) < k:
        if not chosen:
            j = int(np.argmax(base)); chosen.append(j); rest.remove(j); continue
        best, bj = -1e9, rest[0]
        for i in rest:
            div = max(float(cand_vecs[i] @ cand_vecs[c]) for c in chosen)
            val = lam * base[i] - (1 - lam) * div
            if val > best:
                best, bj = val, i
        chosen.append(bj); rest.remove(bj)
    return chosen

RET_COLS = ["citation", "category", "act_title", "act_year", "status",
            "effective_date", "score", "source_snapshot", "full_text"]

def search(query, fetch=FETCH_K, top_k=TOP_K):
    """Hybrid retrieve -> rerank -> MMR. Returns a DataFrame of the top_k sections (or empty)."""
    qv = emb.encode([EMBED_QUERY_PREFIX + query], normalize_embeddings=True)[0].astype("float32")
    dense = tbl.search(qv).limit(fetch).to_pandas()["chunk_id"].tolist()
    try:
        fts = tbl.search(query, query_type="fts").limit(fetch).to_pandas()["chunk_id"].tolist()
    except Exception:
        fts = []                                  # fall back to dense-only if FTS is unavailable
    fused, rrf_map = rrf([dense, fts])
    fused = fused[:fetch]

    cand = chunks_df[chunks_df.chunk_id.isin(fused)].drop_duplicates("unit_id").copy()
    if cand.empty:
        return cand

    try:                                          # reranker is the normal path
        cand["score"] = rr_scores(query, cand["chunk_text"].tolist())
    except Exception as ex:                        # graceful fall-back to fused RRF scores
        print("rerank unavailable -> RRF fallback:", str(ex)[:90])
        cand["score"] = cand["chunk_id"].map(rrf_map).fillna(0.0)

    cand = cand.sort_values("score", ascending=False).head(min(12, len(cand)))
    cv = emb.encode(cand["chunk_text"].tolist(), normalize_embeddings=True).astype("float32")
    order = mmr(cv, cand["score"].to_numpy(), k=top_k)
    out = cand.iloc[order]
    return out[[c for c in RET_COLS if c in out.columns]]

In [9]:
# ── Test the retrieval engine directly (no agent, no LLM) ──────────────────────
_demo = search("can the police arrest me without telling me why", top_k=4)
print(_demo[["citation", "category", "status", "score"]].to_string(index=False))

                                                                       citation          category   status    score
Section 35, Bharatiya Nagarik Suraksha Sanhita, 2023 (in force from 2024-07-01) Criminal & Police in_force 0.880832
                                  Section 29, Northern Indian Ferries Act, 1878 Criminal & Police in_force 0.841907
                                                Section 13, Passports Act, 1967 Criminal & Police in_force 0.861846
                                              Article 22, Constitution of India Criminal & Police in_force 0.832129


## §7 · Evidence sources — legal DB, Wikipedia, web (plain functions, no free tool-calling)

In [10]:
import requests
from concurrent.futures import ThreadPoolExecutor

_io_pool = ThreadPoolExecutor(max_workers=4)

# ── small utils ──────────────────────────────────────────────────────────────
STATE_PREFIXES = (
    "Andhra Pradesh","Arunachal","Assam","Bihar","Chhattisgarh","Goa","Gujarat","Haryana","Himachal",
    "Jharkhand","Karnataka","Kerala","Madhya Pradesh","Maharashtra","Manipur","Meghalaya","Mizoram",
    "Nagaland","Odisha","Orissa","Punjab","Rajasthan","Sikkim","Tamil Nadu","Telangana","Tripura",
    "Uttar Pradesh","Uttarakhand","West Bengal","Jammu","Puducherry","Pondicherry",
)
def _state_note(act_title):
    t = (act_title or "").strip()
    for s in STATE_PREFIXES:
        if t.startswith(s):
            return f" [STATE LAW — appears specific to {s}; may not apply to the user's state]"
    return ""

def _cap(text, n=TOOL_TEXT_CAP):
    text = str(text)
    return text if len(text) <= n else text[:n] + " …[truncated]"

def _clean(v):
    """NaN / None / 'nan' / 'NaT' -> '' ; else a clean string."""
    try:
        if pd.isna(v): return ""
    except (TypeError, ValueError):
        pass
    s = str(v).strip()
    return "" if s.lower() in ("", "nan", "none", "nat", "<na>") else s

def _clamp_topk(k):
    try:
        k = int(k)
    except (TypeError, ValueError):
        return TOP_K
    return max(TOPK_MIN, min(TOPK_MAX, k))

# ── source 1: the central-law database (statute text + citations) ──────────────
def kb_search(query, top_k=None):
    """Returns a list of candidate dicts from the legal database (one row per section)."""
    k = _clamp_topk(top_k) if top_k is not None else TOP_K
    df = search(query.strip(), top_k=k)
    out = []
    if df is not None and len(df):
        for _, r in df.iterrows():
            sc = round(float(r.get("score", 0.0)), 3)
            if sc < CITE_MIN_SCORE:          # cheap pre-filter; the LLM grader is the real gate
                continue
            note = _state_note(r.get("act_title", ""))
            out.append({
                "source": "legal_db",
                "title":  _clean(r.get("citation")),
                "url":    "",
                "text":   (note + "\n" if note else "") + _cap(r.get("full_text", "")),
                "score":  sc,
                "meta":   {"status": _clean(r.get("status")), "effective_date": _clean(r.get("effective_date")),
                           "act_title": _clean(r.get("act_title"))},
            })
    return out

# ── source 2: Wikipedia (plain-language background) ────────────────────────────
_WIKI_API     = "https://en.wikipedia.org/w/api.php"
_WIKI_SUMMARY = "https://en.wikipedia.org/api/rest_v1/page/summary/"
_WIKI_HEADERS = {"User-Agent": "KnowYourRights/1.0 (educational legal-information assistant)"}

def wiki_lookup(query, n=WIKI_MAX_RESULTS):
    """Search Wikipedia and return short extracts (title, url, summary) as candidate dicts."""
    try:
        r = requests.get(_WIKI_API, headers=_WIKI_HEADERS, timeout=WIKI_TIMEOUT,
                         params={"action": "query", "list": "search", "srsearch": query,
                                 "format": "json", "srlimit": n})
        hits = r.json().get("query", {}).get("search", [])
    except Exception:
        return []
    out = []
    for h in hits[:n]:
        title = h.get("title", "")
        slug  = requests.utils.quote(title.replace(" ", "_"))
        try:
            s = requests.get(_WIKI_SUMMARY + slug, headers=_WIKI_HEADERS, timeout=WIKI_TIMEOUT).json()
            extract = (s.get("extract") or "").strip()
            url = (s.get("content_urls", {}).get("desktop", {}).get("page")
                   or f"https://en.wikipedia.org/wiki/{slug}")
        except Exception:
            extract = _re.sub(r"<[^>]+>", "", h.get("snippet", "")).strip()
            url = f"https://en.wikipedia.org/wiki/{slug}"
        if extract:
            out.append({"source": "wikipedia", "title": title, "url": url,
                        "text": extract[:1200], "score": 0.0, "meta": {}})
    return out

# ── source 3: the public web (current procedure, portals, notifications, state law) ──
_web_cache = {}; _web_calls = []

def web_search_core(query, n=WEB_MAX_RESULTS):
    """DuckDuckGo search with a TTL cache + per-minute limit. Returns candidate dicts."""
    q = query.strip(); ck = hashlib.md5(q.lower().encode()).hexdigest()
    hit = _web_cache.get(ck)
    if hit and (time.time() - hit[0] < WEB_CACHE_TTL):
        return hit[1]
    now = time.time(); _web_calls[:] = [t for t in _web_calls if now - t < 60]
    if len(_web_calls) >= WEB_MAX_PER_MIN:
        return []
    try:
        from ddgs import DDGS
        with DDGS(timeout=WEB_TIMEOUT) as d:
            raw = list(d.text(q, max_results=n))
    except Exception:
        return []
    _web_calls.append(time.time())
    out = [{"source": "web", "title": r.get("title", ""), "url": r.get("href", ""),
            "text": (r.get("body", "") or "")[:800], "score": 0.0, "meta": {}} for r in raw[:n]]
    _web_cache[ck] = (time.time(), out)
    return out

SOURCE_FN = {"legal_db": kb_search, "wikipedia": wiki_lookup, "web": web_search_core}

@dataclass
class Candidate:
    id: int
    source: str
    title: str
    url: str
    text: str
    score: float
    meta: dict

print("evidence sources ready · legal_db + wikipedia + web")


evidence sources ready · legal_db + wikipedia + web


In [11]:
# ── Test each evidence source directly (no agent, no LLM) ───────────────────────
print("legal_db:")
for c in kb_search("Right to Information Act 2005 procedure", top_k=3):
    print("   •", c["title"], "| score", c["score"])

print("\nwikipedia (needs Internet On):")
for c in wiki_lookup("Right to Information Act India"):
    print("   •", c["title"], "—", c["url"])

print("\nweb (needs Internet On):")
for c in web_search_core("RTI online application portal India"):
    print("   •", c["title"], "—", c["url"])


legal_db:
   • Section 33, Indian Institutes of Management Act, 2017 | score 0.928
   • Section 7, Right to Information Act, 2005 | score 0.861
   • Section 2, Right to Information Act, 2005 | score 0.92

wikipedia (needs Internet On):
   • Right to Information Act, 2005 — https://en.wikipedia.org/wiki/Right_to_Information_Act%2C_2005
   • Central Information Commission — https://en.wikipedia.org/wiki/Central_Information_Commission

web (needs Internet On):
   • RTI Online :: Home | Submit RTI Request | Submit RTI First ... — https://rtionline.gov.in/
   • File your RTI application online | National Government Services Portal — https://services.india.gov.in/service/detail/file-your-rti-application-online-1
   • RTI Portal | Home — https://rtiportal.kerala.gov.in/


## §8 · Structured contracts between the stages (Plan / Grades)

In [12]:
from pydantic import BaseModel, Field
from typing import Literal, List

# Structured "contracts" between the stages. We get the robustness of structured outputs in a
# provider-agnostic way: each agent is told to emit ONLY this JSON, and we validate with Pydantic
# (so it works even if a given NIM model doesn't support strict json_schema response_format).

class ResearchStep(BaseModel):
    tool: Literal["legal_db", "wikipedia", "web"]
    query: str
    reason: str = ""

class Plan(BaseModel):
    kind: Literal["smalltalk", "capability", "legal_question", "out_of_scope"]
    answer_kind: Literal["definition", "procedure", "rights", "punishment", "mixed", "none"] = "none"
    normalized_query: str = ""
    steps: List[ResearchStep] = Field(default_factory=list)

class SourceGrade(BaseModel):
    id: int
    relevant: bool
    note: str = ""

class Grades(BaseModel):
    grades: List[SourceGrade] = Field(default_factory=list)

def _extract_json(text):
    """Pull the first balanced JSON object/array out of a model reply (strips ``` fences / prose)."""
    s = str(text).strip()
    if s.startswith("```"):
        s = _re.sub(r"^```[a-zA-Z]*\n?", "", s)
        s = _re.sub(r"\n?```$", "", s).strip()
    for open_c, close_c in (("{", "}"), ("[", "]")):
        i = s.find(open_c)
        if i != -1:
            depth = 0
            for j in range(i, len(s)):
                if s[j] == open_c: depth += 1
                elif s[j] == close_c:
                    depth -= 1
                    if depth == 0:
                        return s[i:j + 1]
    return s

async def run_json(agent, user_input, model_cls, default, retries=1):
    """Run a structured-output agent, validate its JSON with Pydantic, retry once, else return default."""
    msg = user_input
    for attempt in range(retries + 1):
        try:
            res = await Runner.run(agent, msg, max_turns=2)
            raw = res.final_output if isinstance(res.final_output, str) else str(res.final_output)
            return model_cls.model_validate_json(_extract_json(raw))
        except Exception as e:
            if attempt < retries:
                msg = (user_input + "\n\nIMPORTANT: reply with ONLY the JSON object described above — "
                       "no markdown fences, no commentary.")
                continue
            print("  (structured parse failed -> safe default):", str(e)[:120])
            return default

print("schemas ready · Plan / Grades + run_json()")


schemas ready · Plan / Grades + run_json()


## §9 · The four agents — planner · grader · writer · concierge

In [13]:
# Four small, single-purpose agents. Each prompt is short and focused — the opposite of one giant
# system prompt. The planner & grader return JSON (parsed via run_json); the writer & concierge
# stream plain prose.

PLANNER_PROMPT = """
You are the planner for KnowYourRights, an assistant on CENTRAL Indian law (the Constitution, central
Acts, and the 2024 BNS / BNSS / BSA codes that replaced the IPC / CrPC / Evidence Act).
Classify the user's latest message and, if it is a legal question, plan the research to answer it well.

Return ONLY a JSON object of this exact shape (no markdown, no prose):
{
  "kind": "smalltalk" | "capability" | "legal_question" | "out_of_scope",
  "answer_kind": "definition" | "procedure" | "rights" | "punishment" | "mixed" | "none",
  "normalized_query": "<the question restated in clear English legal terms; empty if not legal>",
  "steps": [ {"tool": "legal_db" | "wikipedia" | "web", "query": "<search query>", "reason": "<why>"} ]
}

kind:
- smalltalk      : greetings, thanks, chit-chat (e.g. "hi bro", "thanks!").
- capability     : asking what you can do.
- legal_question : anything about Indian law, rights, procedures, penalties, legal documents.
- out_of_scope   : not about law (e.g. "write me a poem", "what's the weather").

For a legal_question:
- normalized_query: ALWAYS expand acronyms — RTI -> "Right to Information Act, 2005"; FIR -> "First
  Information Report"; NDPS -> "Narcotic Drugs and Psychotropic Substances Act, 1985"; IPC/CrPC -> the
  2024 BNS / BNSS.
- answer_kind: the shape that best fits (definition / procedure / rights / punishment / mixed).
- steps (1-5): for EACH part of the question pick the BEST source and a precise query —
    legal_db  : what the statute actually says (sections, exact citations). Use the FULL Act name.
    wikipedia : plain-language background, "what is X", how something generally works.
    web       : current official PROCEDURE, government PORTALS and links, fees, notifications, recent
                amendments, or STATE-specific law (the database is central-only and as-of a snapshot).
  Be deliberate: a "how do I file / apply / register X" question MUST include a `web` step to fetch the
  official portal and steps; "what is X" should include `wikipedia`; "what does the law say / what is
  the punishment" should include `legal_db`. Mix sources when that serves the question.
For smalltalk / capability / out_of_scope: set normalized_query="", answer_kind="none", steps=[].
""".strip()

GRADER_PROMPT = """
You are a strict relevance grader for a legal assistant. You are given the user's QUESTION and a
numbered list of candidate SOURCES (law sections, Wikipedia extracts, web results). Decide which
candidates genuinely help answer THIS question.

Return ONLY a JSON object of this exact shape (no markdown, no prose):
{ "grades": [ {"id": <int>, "relevant": true|false, "note": "<short reason>"} ] }

Rules:
- relevant=true ONLY if the source is actually about the question's topic and helps answer it.
- Be strict: a source that merely shares a word (e.g. some unrelated "... Act, 20XX", or an
  institution's own statute) but is NOT about the topic is relevant=false.
- It is fine — and common — to mark several as false.
- Return exactly one grade for EVERY candidate id you are given.
""".strip()

WRITER_PROMPT = """
You are KnowYourRights, explaining CENTRAL Indian law to ordinary people in clear, plain language.
You give legal INFORMATION, not legal advice.

You are given the user's QUESTION, an ANSWER-SHAPE hint, and a set of VETTED SOURCES. Only the vetted
sources are trustworthy: do not rely on outside knowledge for specific legal claims (section numbers,
procedures, figures), and NEVER cite anything that is not in the vetted set.

Write the most useful answer for THIS question, choosing the format that fits it:
- "definition": a short, clear paragraph (no bullet lists).
- "procedure" / "how do I...": concrete, ordered steps, and include the official portal/link if a
  vetted web source provides one.
- "rights" / "punishment": explain it plainly first, then the specifics.
Do NOT force bullet points or a fixed template; many answers are best as a couple of short paragraphs.
Be concise and direct — match the length to the question.

Cite specific provisions inline by name, e.g. "Section 6, Right to Information Act, 2005", drawn ONLY
from the vetted sources; include an official link if a vetted web source gives one. If the vetted
sources do NOT contain a precise provision for part of the question, say so honestly and point the user
to the right authority or official site — do not invent a provision. Reply in the user's language.
Do NOT add a legal-advice disclaimer; the interface already shows one.
""".strip()

CONCIERGE_PROMPT = """
You are KnowYourRights, a friendly assistant for questions about Indian law. Reply briefly and warmly
in the user's language, in 1-2 sentences, with no bullet lists and no legal-advice disclaimer.
- Greeting / chit-chat: greet back and invite a legal question, naturally.
- "What can you do": say you explain people's rights and the laws behind them (police & arrest, RTI,
  consumer, work & salary, tenancy, and more), point to the exact section, and can check official
  sources for current procedures.
- Not about law: gently say that's outside what you help with, and offer to take an Indian-law question.
Keep it natural; never dump a feature list.
""".strip()

planner_agent   = Agent(name="Planner",   instructions=PLANNER_PROMPT,   model=llm_model,
                        model_settings=ModelSettings(temperature=PLAN_TEMP,  retry=RETRY_SETTINGS))
grader_agent    = Agent(name="Grader",    instructions=GRADER_PROMPT,    model=llm_model,
                        model_settings=ModelSettings(temperature=GRADE_TEMP, retry=RETRY_SETTINGS))
writer_agent    = Agent(name="Writer",    instructions=WRITER_PROMPT,    model=llm_model,
                        model_settings=ModelSettings(temperature=WRITER_TEMP, max_tokens=WRITER_MAX_TOKENS,
                                                     retry=RETRY_SETTINGS))
concierge_agent = Agent(name="Concierge", instructions=CONCIERGE_PROMPT, model=llm_model,
                        model_settings=ModelSettings(temperature=CONCIERGE_TEMP, max_tokens=300,
                                                     retry=RETRY_SETTINGS))

print("agents ready ·", ", ".join(a.name for a in (planner_agent, grader_agent, writer_agent, concierge_agent)))


agents ready · Planner, Grader, Writer, Concierge


## §10 · The pipeline — plan → retrieve → grade → write

In [14]:
def _history_block(history, n=4):
    """Compact recent turns so the planner can resolve references ('it', 'that fine') and the writer
    can stay coherent across a conversation."""
    if not history:
        return ""
    lines = [f"{m['role'].upper()}: {m['content']}" for m in history[-n:] if m.get("content")]
    return "RECENT CONVERSATION:\n" + "\n".join(lines) + "\n\n" if lines else ""

# ── Stage 1: plan (classify intent + plan research) ────────────────────────────
async def make_plan(user_msg, history=None):
    default = Plan(kind="legal_question", answer_kind="mixed", normalized_query=user_msg,
                   steps=[ResearchStep(tool="legal_db", query=user_msg, reason="fallback")])
    return await run_json(planner_agent, f"{_history_block(history)}USER MESSAGE: {user_msg}", Plan, default)

# ── Stage 2: retrieve (run the planned steps in parallel) ──────────────────────
async def gather_evidence(plan, ctx):
    loop = asyncio.get_event_loop()
    async def run_step(step):
        fn = SOURCE_FN.get(step.tool)
        if fn is None:
            return []
        ctx.tracker.record_call(step.tool)
        try:
            return await loop.run_in_executor(_io_pool, fn, step.query)
        except Exception:
            ctx.tracker.record_error(step.tool); return []
    results = await asyncio.gather(*[run_step(s) for s in plan.steps])
    cands, seen = [], set()
    for chunk in results:
        for item in chunk:
            key = (item["source"], item["title"], item["url"])
            if key in seen:
                continue
            seen.add(key)
            cands.append(Candidate(id=len(cands), **item))
    return cands

# ── Stage 3: grade (LLM keeps only genuinely relevant sources -> fixes junk citations) ──
async def grade_sources(question, candidates):
    if not candidates:
        return set()
    listing = "\n".join(f"[{c.id}] ({c.source}) {c.title}\n    {c.text[:400]}" for c in candidates)
    default = Grades(grades=[SourceGrade(id=c.id, relevant=True) for c in candidates])  # only used on parse failure
    g = await run_json(grader_agent, f"QUESTION: {question}\n\nCANDIDATES:\n{listing}", Grades, default)
    return {gr.id for gr in g.grades if gr.relevant}

# ── Stage 4: write (compose the answer from vetted sources; format follows the question) ──
def _vetted_block(vetted):
    if not vetted:
        return "(no vetted sources were found — be honest about this and point to the right authority)"
    blocks = []
    for c in vetted:
        tag = {"legal_db": "LAW", "wikipedia": "WIKIPEDIA", "web": "WEB"}[c.source]
        link = f"  (official link: {c.url})" if c.url else ""
        blocks.append(f"[{tag}] {c.title}{link}\n{c.text[:900]}")
    return "\n\n".join(blocks)

def build_writer_input(user_msg, plan, vetted, history=None):
    return (f"{_history_block(history)}QUESTION: {user_msg}\n"
            f"ANSWER SHAPE HINT: {plan.answer_kind}\n\n"
            f"VETTED SOURCES (use ONLY these for specific legal claims):\n{_vetted_block(vetted)}")

async def concierge_reply(user_msg, history=None):
    try:
        res = await Runner.run(concierge_agent, f"{_history_block(history)}USER: {user_msg}", max_turns=2)
        return res.final_output
    except Exception:
        return "Hi! Ask me anything about your rights under Indian law."

def _report(before, after, ctx):
    d = _diff(before["calls"], after["calls"])
    print("  " + "-" * 58)
    print("  retrieval: " + (", ".join(f"{k}×{v}" for k, v in d.items()) if d else "none") +
          f"  |  LLM calls: {after['llm_calls'] - before['llm_calls']}")
    if ctx.sources:
        print("  kept: " + ", ".join(c.title for c in ctx.sources[:6]))
    print("  " + "=" * 58)

async def respond(user_msg, history=None, verbose=True):
    """One full turn (non-streaming; used by the tests below). The Gradio UI uses respond_stream()."""
    ctx = AppContext(tracker=TRACKER); before = TRACKER.snapshot()
    if verbose:
        print(f"\n{'='*60}\nUSER: {user_msg}")
    plan = await make_plan(user_msg, history)
    if verbose:
        print(f"  plan: {plan.kind} / {plan.answer_kind} | steps={[(s.tool, s.query) for s in plan.steps]}")
    if plan.kind in ("smalltalk", "capability", "out_of_scope") or not plan.steps:
        ans = await concierge_reply(user_msg, history)
        if verbose: _report(before, TRACKER.snapshot(), ctx)
        return ans
    cands = await gather_evidence(plan, ctx)
    keep  = await grade_sources(plan.normalized_query or user_msg, cands)
    ctx.sources = [c for c in cands if c.id in keep]
    if verbose:
        print(f"  retrieved {len(cands)} candidate(s) -> kept {len(ctx.sources)} relevant")
    try:
        res = await Runner.run(writer_agent, build_writer_input(user_msg, plan, ctx.sources, history), max_turns=2)
        ans = res.final_output
    except RateLimitError:
        TRACKER.record_error("rate_limit")
        ans = (f"The AI service is rate-limited right now (NIM free tier ≈40 req/min; retried {RETRY_MAX}× "
               f"with backoff). Please wait ~{RETRY_FINAL_SLEEP}s and try again.")
    except Exception as e:
        TRACKER.record_error("unexpected")
        ans = f"Sorry — something went wrong while writing the answer: {str(e)[:200]}"
    if verbose:
        _report(before, TRACKER.snapshot(), ctx)
    return ans

print("pipeline ready · make_plan -> gather_evidence -> grade_sources -> write  (respond())")


pipeline ready · make_plan -> gather_evidence -> grade_sources -> write  (respond())


In [15]:
# ── Test the full pipeline: a greeting, then two real questions ─────────────────
print(await respond("hi bro"))                       # -> concierge, no tools

print("\n" + await respond("what are my rights if police arrest me without a warrant?"))

print("\n" + await respond("how can I file an RTI and what is it?"))   # -> wiki + legal_db + web; portal link

TRACKER.report()



USER: hi bro
  plan: smalltalk / none | steps=[]
  ----------------------------------------------------------
  retrieval: none  |  LLM calls: 2
Hey! What's up? Feel free to ask me any Indian‑law question you have.

USER: what are my rights if police arrest me without a warrant?
  plan: legal_question / rights | steps=[('legal_db', 'BNSS 2024 Section 41 arrest without warrant rights'), ('legal_db', 'Constitution of India Article 22 protections against arrest and detention'), ('web', 'rights of person arrested without warrant India police procedure official portal')]
  retrieved 13 candidate(s) -> kept 4 relevant
  ----------------------------------------------------------
  retrieval: legal_db×2, web×1  |  LLM calls: 3
  kept: Section 47, Bharatiya Nagarik Suraksha Sanhita, 2023 (in force from 2024-07-01), Article 22, Constitution of India, CRPC | Know your rights, Know your Right if you Arrested - NCIB

If the police arrest you **without a warrant**, the law gives you several specifi

## §11 · Gradio chat — streaming, staged thinking, and a citations panel

In [16]:
import gradio as gr
from openai.types.responses import ResponseTextDeltaEvent

_STEP_LABELS = {"legal_db": "🔎 Searching the legal database",
                "wikipedia": "📖 Reading Wikipedia",
                "web": "🌐 Searching the web"}
_SRC_HINT = "_Ask a question to see the laws I cite._"

def _sources_md(ctx):
    if not ctx.sources:
        return "_No strongly-relevant source was used for this answer._"
    law  = [c for c in ctx.sources if c.source == "legal_db"]
    wiki = [c for c in ctx.sources if c.source == "wikipedia"]
    web  = [c for c in ctx.sources if c.source == "web"]
    out = []
    if law:
        out.append("**From the legal database (authoritative):**")
        for c in law:
            st = c.meta.get("status", "")
            badge = "🟢 in force" if st == "in_force" else ("⚪ omitted" if st == "omitted" else (st or "—"))
            eff = (c.meta.get("effective_date", "") or "").strip()
            eff = f" · effective {eff}" if eff and eff.lower() not in ("none", "nat", "nan") else ""
            out.append(f"- **{c.title}** — {badge}{eff}")
    if wiki:
        out.append(("\n" if out else "") + "**Background (Wikipedia):**")
        for c in wiki:
            out.append(f"- [{c.title}]({c.url})")
    if web:
        out.append(("\n" if out else "") + "**From the web (please verify):**")
        for c in web:
            out.append(f"- [{c.title}]({c.url})")
    return "\n".join(out)

async def _stream_text(agent, agent_input, answer, view, sources_md, fallback):
    """Stream one agent's prose into `answer['content']`, yielding the growing view. Returns nothing."""
    acc = ""
    res = Runner.run_streamed(agent, agent_input, max_turns=2)
    async for ev in res.stream_events():
        if ev.type == "raw_response_event" and isinstance(ev.data, ResponseTextDeltaEvent) and ev.data.delta:
            acc += ev.data.delta; answer["content"] = acc
            yield view(), sources_md()
    if not acc:
        answer["content"] = res.final_output or fallback
        yield view(), sources_md()

async def respond_stream(user_msg, history):
    """Async generator -> yields (turn_messages, sources_md). Shows the real pipeline stages as
    collapsible 'thinking' steps, then streams the answer, then fills the sources panel."""
    ctx = AppContext(tracker=TRACKER); before = TRACKER.snapshot()
    steps, answer = [], {"role": "assistant", "content": ""}
    def view():
        return steps + ([answer] if answer["content"] else [])
    def add_step(title, body="…"):
        steps.append({"role": "assistant", "content": body, "metadata": {"title": title}})

    add_step("🧭 Understanding your question")
    yield view(), _SRC_HINT
    plan = await make_plan(user_msg, history)

    # chit-chat / capability / out-of-scope -> concierge, no thinking bubbles
    if plan.kind in ("smalltalk", "capability", "out_of_scope") or not plan.steps:
        steps.clear()
        try:
            async for v in _stream_text(concierge_agent, f"{_history_block(history)}USER: {user_msg}",
                                        answer, view, lambda: "_No sources needed._",
                                        "Hi! How can I help with your legal question?"):
                yield v
        except Exception:
            answer["content"] = "Hi! Ask me anything about your rights under Indian law."
            yield view(), "_No sources needed._"
        _report(before, TRACKER.snapshot(), ctx); return

    # legal question: show a step per planned source, then retrieve + grade
    for s in plan.steps:
        add_step(_STEP_LABELS.get(s.tool, f"🔧 {s.tool}"), f"`{s.query}`")
    yield view(), _SRC_HINT
    cands = await gather_evidence(plan, ctx)
    keep  = await grade_sources(plan.normalized_query or user_msg, cands)
    ctx.sources = [c for c in cands if c.id in keep]
    add_step("✅ Selecting relevant sources", f"kept {len(ctx.sources)} of {len(cands)}")
    yield view(), _sources_md(ctx)

    # write the answer (streamed)
    try:
        async for v in _stream_text(writer_agent, build_writer_input(user_msg, plan, ctx.sources, history),
                                    answer, view, lambda: _sources_md(ctx),
                                    "I couldn't compose an answer — please try rephrasing."):
            yield v
    except RateLimitError:
        TRACKER.record_error("rate_limit")
        answer["content"] = (f"Rate-limited right now (NIM free tier ≈40 req/min). Please wait "
                             f"~{RETRY_FINAL_SLEEP}s and try again.")
        yield view(), _sources_md(ctx)
    except Exception as e:
        TRACKER.record_error("unexpected")
        answer["content"] = f"Sorry — something went wrong: {str(e)[:200]}"
        yield view(), _sources_md(ctx)
    _report(before, TRACKER.snapshot(), ctx)

# ── Gradio wiring ────────────────────────────────────────────────────────────────
async def _gr_respond(message, history):
    if not message or not message.strip():
        yield history, "", gr.update(); return
    base = (history or []) + [{"role": "user", "content": message.strip()}]
    yield base, "", "_Working…_"
    async for turn_msgs, sources in respond_stream(message.strip(), history or []):
        yield base + turn_msgs, "", sources

def _gr_reset():
    return [], "", _SRC_HINT

DISCLAIMER = (
    "**KnowYourRights** shares general information about *central* Indian law "
    "(the Constitution, central Acts, and the 2024 BNS / BNSS / BSA codes), with citations. "
    "It is **not a lawyer** and does **not** give legal advice. Always verify recent changes."
)
EXAMPLES = [
    "What are my rights if the police arrest me without a warrant?",
    "How do I file an RTI application, and what is it?",
    "What's the punishment for possession of cannabis, and what quantity counts as small?",
    "Police ne mujhe bina warrant ke arrest kar liya, kya yeh legal hai?",
]

with gr.Blocks(title="KnowYourRights") as demo:
    gr.Markdown("# ⚖️ KnowYourRights — Indian Legal Rights Assistant")
    gr.Markdown(DISCLAIMER)
    chat = gr.Chatbot(height=460, buttons=["copy"])
    with gr.Accordion("📚 Sources & citations", open=True):
        sources_md = gr.Markdown(_SRC_HINT)
    with gr.Row():
        box  = gr.Textbox(placeholder="Ask about your legal rights (English / Hindi / Hinglish)…",
                          scale=5, show_label=False)
        send = gr.Button("Send", variant="primary", scale=1)
    clear = gr.Button("🗑️ New conversation", variant="secondary")
    gr.Examples(examples=EXAMPLES, inputs=box)
    gr.Markdown("_⚖️ General legal information, not legal advice. For your situation, consult a qualified lawyer._")

    for ev in (send.click, box.submit):
        ev(_gr_respond, inputs=[box, chat], outputs=[chat, box, sources_md])
    clear.click(_gr_reset, outputs=[chat, box, sources_md])

print("Gradio UI built (staged thinking + streamed answer + grouped sources) — launch with the next cell.")


Gradio UI built (staged thinking + streamed answer + grouped sources) — launch with the next cell.


In [18]:
gr.close_all()


Closing server running on port: 7861


In [17]:
# Gradio 6: theme/css/js are passed to launch(), not Blocks().
demo.launch(theme=gr.themes.Soft())
# On Kaggle, use a public URL instead:
# demo.launch(theme=gr.themes.Soft(), share=True)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


  ----------------------------------------------------------
  retrieval: none  |  LLM calls: 2
  ----------------------------------------------------------
  retrieval: wikipedia×1  |  LLM calls: 3
rerank unavailable -> RRF fallback:rerank unavailable -> RRF fallback: CUDA out of memory. Tried to allocate 76.00 MiB. GPU 0 has a total capacity of 4.00 GiB of
 CUDA out of memory. Tried to allocate 118.00 MiB. GPU 0 has a total capacity of 4.00 GiB o
  ----------------------------------------------------------
  retrieval: legal_db×2, web×1  |  LLM calls: 3
  kept: Welcome to Human Rights Commissions Network, How to file an online complaint, Register online complaints with National Human Rights Commission | National Government Services Portal
  ----------------------------------------------------------
  retrieval: legal_db×1, web×3, wikipedia×1  |  LLM calls: 3
  kept: Welcome to Delhi Police Online Complaint Lodging System - OTP, Delhi Police Online Complaint Portal: Documents & Mobile

## Design notes — why it's built this way

**One question, four small agents, coordinated by code.** Instead of a single agent with a long system
prompt deciding everything, this notebook is a **workflow** (in Ed's sense — deterministic Python
orchestration, *not* handoffs): `make_plan → gather_evidence → grade_sources → write`. Each agent has a
short, single-purpose prompt, which a 70B model follows far more reliably than one ~600-word rulebook.

**Stage 1 · Planner (structured output).** Classifies the message — `smalltalk` / `capability` /
`legal_question` / `out_of_scope` — and, for a legal question, plans the research: it expands acronyms
("RTI" → "Right to Information Act, 2005"), picks an `answer_kind`, and chooses, per sub-question, the
best source. This is how the model **decides the scope and shape of the reply itself** — and how
greetings are handled by *reasoning*, not by a hardcoded string list ("hi bro" → `smalltalk`).

**Stage 2 · Retrieve (parallel, plain functions).** The plan's steps are dispatched to ordinary async
functions — `legal_db` (statute text + citations), `wikipedia` (plain-language background), `web`
(official portals, fees, notifications, recent or state-specific law) — run together with
`asyncio.gather`. The LLM never freely "calls a tool", so there is **no tool-call hallucination**: it
emits a validated plan and code executes it.

**Stage 3 · Grade (structured output).** An LLM judges each retrieved candidate for genuine relevance
and we keep only the ones it passes. This is what fixes the junk-citation problem: a numeric rerank
score can't tell "about RTI" from "merely mentions an Act", but a grader can — so the IIM Act / unrelated
university statutes are dropped from both the answer and the Sources panel.

**Stage 4 · Write (no fixed template).** The writer composes the answer from **only the vetted sources**,
choosing the format that fits the question (a definition is a short paragraph; a "how do I file…" is
concrete steps with the official link). It cites provisions by name drawn only from the vetted set, and
abstains honestly when nothing precise was found — so it can't fabricate a citation.

**Trust by construction.** Citations shown in the panel are exactly the vetted sources the writer used —
grader-approved, decoupled from the model's prose. Acronyms are expanded before search; statute,
background, and current-procedure questions each go to the right source; the disclaimer lives in the UI,
not in every message.

**Robustness choices.** Structured outputs (Plan, Grades) are validated with Pydantic via `run_json()`
rather than a provider-specific `response_format`, so the pipeline works even if a given NIM model
doesn't support strict json_schema; each structured stage retries once then degrades to a safe default.
The proactive RPM throttle + the SDK's header-aware retry/backoff handle NIM's free-tier rate limit; a
legal question is ~3 LLM calls (plan + grade + write), chit-chat ~2 (plan + reply).

**Tuning knobs (all in §1):** sources & relevance — `FETCH_K`/`TOP_K`/`CITE_MIN_SCORE`/`MMR_LAMBDA`,
`WEB_*`, `WIKI_*`; agents — `PLAN_TEMP`/`GRADE_TEMP`/`WRITER_TEMP`/`WRITER_MAX_TOKENS`; rate limits —
`NIM_RPM`/`RETRY_*`. The reranker is swappable; the embedder (`BAAI/bge-m3`) is locked to the database.
